# Phase 3 — Candidate generation validation

This notebook builds and evaluates the blocking pipeline in `src/candidate_generation.py`. For each S1 record it produces a manageable set of S2/S3 candidates, without ever forming S1 × S2 or S1 × S3.

**Rules followed here**
* Training data only.
* The ground truth is used **only** to measure recall. Keys and frequencies come from the source records.
* DuckDB does every join, capped at 1.5 GB of memory with a 2 GB spill limit. Features are stored as Parquet.
* Only candidates for the evaluated S1 sample are ever materialised.
* Every strategy has frequency caps and a raw-pair explosion guard.

**Sections**
1. Setup and resource checks
2. Streaming normalisation to Parquet (reused if already built)
3. Blocking statistics
4. 10k pilot with the initial configuration: every strategy
5. Failure analysis of the pilot
6. Parameter sweep (marginal recall per extra candidate)
7. Final configuration on the 10k sample, with leave-one-out and greedy selection
8. 100k validation (chunked)
9. Full-scale execution path and estimate
10. Save results

## 1. Setup

In [1]:
import sys, os, time, json, shutil, threading
from pathlib import Path
from dataclasses import replace, asdict
import duckdb, pandas as pd, psutil
from IPython.display import display

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT)); os.chdir(PROJECT)
import src.candidate_generation as C
from src.candidate_generation import (BlockingConfig, STRATEGY_BITS as B, DEFAULT_STRATEGIES, CAP_PRIORITY,
                                      evaluate_candidate_recall, generate_candidates, generate_candidates_chunked)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)

RAW = PROJECT / "data" / "raw"; P = PROJECT / "data" / "processed" / "phase3"; EXP = PROJECT / "experiments"
P.mkdir(parents=True, exist_ok=True)
GT_TSV = RAW / "train_ground_truth.tsv"
MIN_FREE_GB = 4.0
RESULTS = {"notebook_started": time.strftime("%Y-%m-%d %H:%M:%S")}

class PeakRSS:
    """Tracks peak RSS of this process (+children) in a background thread."""
    def __enter__(self):
        self.peak, self._stop = 0, False
        def run():
            p = psutil.Process()
            while not self._stop:
                try: self.peak = max(self.peak, p.memory_info().rss + sum(c.memory_info().rss for c in p.children(recursive=True)))
                except Exception: pass
                time.sleep(0.2)
        self._t = threading.Thread(target=run, daemon=True); self._t.start(); return self
    def __exit__(self, *a):
        self._stop = True; self._t.join()
    @property
    def gb(self): return round(self.peak / 1e9, 2)

vm = psutil.virtual_memory()
free = C.check_disk(P, MIN_FREE_GB)
print(f"duckdb {duckdb.__version__} | CPUs {os.cpu_count()} | RAM total {vm.total/1e9:.1f} GB, available {vm.available/1e9:.1f} GB | free disk {free:.1f} GB")
RESULTS["environment"] = {"ram_total_gb": round(vm.total/1e9, 1), "ram_available_gb_at_start": round(vm.available/1e9, 1),
                          "free_disk_gb_at_start": round(free, 1), "cpus": os.cpu_count(), "duckdb": duckdb.__version__}

duckdb 1.5.5 | CPUs 8 | RAM total 8.6 GB, available 1.7 GB | free disk 6.5 GB


## 2. Streaming normalisation to Parquet
Each source TSV is read by DuckDB in 100k-row batches and normalised by 4 worker processes using the Phase 2 functions (`blocking_record`). Each batch is written as a zstd Parquet part.

Only integer ids and blocking fields are stored; raw text stays in the TSVs. The step is skipped when `_SUCCESS` markers exist.

In [2]:
with PeakRSS() as pk:
    mat = {f"s{i}": C.materialize_blocking_features(RAW / f"train_source{i}.tsv", P / f"features_s{i}", workers=4,
                                                    min_free_gb=MIN_FREE_GB) for i in (1, 2, 3)}
prev = P / "materialize_stats.json"
if all(v.get("skipped") for v in mat.values()) and prev.exists():
    mat = json.loads(prev.read_text()); print("(timings from the original materialisation run)")
else:
    mat["peak_rss_gb"] = pk.gb; prev.write_text(json.dumps(mat, indent=2))
sizes = {f"s{i}": round(sum(f.stat().st_size for f in (P / f"features_s{i}").glob("*.parquet")) / 1e6, 1) for i in (1, 2, 3)}
display(pd.DataFrame({k: v for k, v in mat.items() if k.startswith("s")}).T.assign(parquet_mb=pd.Series(sizes)))
print("peak RSS during materialisation:", mat.get("peak_rss_gb"), "GB")
RESULTS["materialisation"] = {**mat, "parquet_mb": sizes}

skip features_s1: already materialised
skip features_s2: already materialised
skip features_s3: already materialised


(timings from the original materialisation run)


,rows,parts,seconds,bytes,parquet_mb
s1,2206821.0,23.0,76.7,163187963.0,163.2
s2,5034616.0,51.0,201.8,384414225.0,384.4
s3,5285603.0,53.0,211.4,401591475.0,401.6


peak RSS during materialisation: 1.03 GB


In [3]:
con = C.connect(P / "blocking_stats.duckdb", memory_limit="1500MB", threads=4, temp_dir=P / "duckdb_tmp", max_temp="2GB")
C.register_features(con, P)
print("rows:", con.execute("SELECT (SELECT COUNT(*) FROM feat_s1), (SELECT COUNT(*) FROM feat_t)").fetchone())
display(con.execute("SELECT * FROM feat_t USING SAMPLE 6 ROWS (reservoir, 11)").df())

rows: (2206821, 10320219)


,id,src,country,name_key,name_keys,name_core,name_tokens,name_phon,state,hn,street,places,addr_tokens,numbers
0,2759979128,2,us,regionaledison,regionaledison,regional edison,regional edison,atsn rknl,oh,,antigua dr,columbus,antigua columbus,
1,2140308377,2,us,dentalgreen,dentalgreen,dental green,dental green,krn tntl,ny,3253,7th ave,troy,ave troy,3253 7
2,2346604707,2,us,precisionbrandslogistics,precisionbrandslogistics,precision brands logistics,precision brands logistics,lkstks prksn prnts,wa,136,st,seattle,seattle,136
3,2640493688,2,india,bgtechnology,bgtechnology,bg technology,bg technology,pk tknlk,kl,23,,jaleel bldg|nr government hospital|ottappalam,jaleel bldg government hospital ottappalam,23 245 3
4,2905409533,2,india,arbansolyushans,arbansolyushans,arban solyushans,arban solyushans,arpn slsns,gj,195,shankdi sheri,taluka kalol|district gandhinagar|balva|gandhinagar|gand...,shankdi sheri taluka kalol district gandhinagar balva ga...,195
5,2538981952,2,us,hanleymillnerriverside,hanleymillnerriverside,hanley and millner riverside,hanley millner riverside,ant hnl mlnr rprst,co,320,rio grande ave,alamosa e,rio grande ave alamosa,320


## 3. Blocking statistics (target-side key frequencies; no ground truth)
Existing tables are reused. Only the no-state rows and the n-gram table depend on configuration.

The **compound** strategy has no global table. Its keys (state × house number or place × phonetic token) would number in the tens of millions, so their frequencies are computed per query chunk, for that chunk's keys only.

In [4]:
have = {r[0] for r in con.execute("SELECT table_name FROM duckdb_tables() WHERE database_name = current_database()").fetchall()}
need = {"name": "st_name", "phonetic": "st_phonetic", "structured": "st_hn", "rare_name": "st_rare_name",
        "rare_phon": "st_rare_phon", "rare_addr": "st_rare_addr", "ngram": "st_ngram"}
missing = [k for k, t in need.items() if t not in have]
with PeakRSS() as pk:
    t = time.time(); st = C.build_blocking_statistics(con, BlockingConfig(), kinds=missing) if missing else {"seconds": {}}
print("built:", missing or "nothing (reused)", f"in {time.time()-t:.0f}s | peak RSS {pk.gb} GB")
stats_rows = {r[0]: r[1] for r in con.execute("SELECT table_name, estimated_size FROM duckdb_tables() WHERE table_name LIKE 'st_%' ORDER BY 1").fetchall()}
display(pd.Series(stats_rows, name="rows").to_frame().T)
print("Heaviest exact name keys (targets per key) — why a national cap is needed:")
display(con.execute("SELECT country, k, n FROM st_name ORDER BY n DESC LIMIT 8").df().T)
print("Heaviest (state, house number) blocks — why structured keys need refinement:")
display(con.execute("SELECT country, state, hn, n FROM st_hn ORDER BY n DESC LIMIT 8").df().T)
print("Most frequent name tokens per (country, state) — never used as rare tokens:")
display(con.execute("SELECT country, state, k, n FROM st_rare_name ORDER BY n DESC LIMIT 8").df().T)
RESULTS["statistics_tables_rows"] = stats_rows
RESULTS["statistics_build_seconds"] = st["seconds"]

built: nothing (reused) in 0s | peak RSS 0.16 GB


,st_hn,st_hn_refined,st_name,st_name_state,st_ngram,st_phonetic,st_phonetic_state,st_rare_addr,st_rare_name,st_rare_phon
rows,490646,4727469,5320348,84106,1048593,4237433,255346,993848,2591062,1297749


Heaviest exact name keys (targets per key) — why a national cap is needed:


,0,1,2,3,4,5,6,7
country,us,us,us,us,us,us,us,us
k,meridian,earnosethroat,physicaltherapy,primarycare,pediatricdental,womenshealth,familycenter,behavioralhealth
n,1695,1600,1368,1278,1261,1255,1249,1196


Heaviest (state, house number) blocks — why structured keys need refinement:


,0,1,2,3,4,5,6,7
country,india,india,india,india,india,india,india,india
state,mh,mh,mh,dl,mh,dl,dl,tg
hn,1,3,2,1,4,2,3,1
n,34679,28458,27064,25092,19152,18878,17825,16530


Most frequent name tokens per (country, state) — never used as rare tokens:


,0,1,2,3,4,5,6,7
country,india,india,india,india,us,india,us,india
state,mh,mh,dl,dl,tx,mh,tx,mh
k,india,services,delhi,india,center,center,partners,limited
n,47601,34460,33774,29855,28237,27289,26803,25552


## 4. 10k pilot with the initial configuration
The pilot uses a deterministic sample: `hash(id) % 220 = 0` gives about 10k S1 records. Candidates are drawn from **all** 10.3M targets, so the candidate counts are realistic.

The initial configuration is the first design: tight caps, no compound strategy and no no-state fallback.

In [5]:
INITIAL = BlockingConfig(phon_cap=50, phon_state_cap=50, hn_cap=50, hn_refined_cap=50, addr_token_df_cap=50,
                         ngram_m=6, ngram_df_cap=300, stateless_fallback=False, per_s1_cap=None)
INITIAL_STRATEGIES = ["name", "phonetic", "structured", "rare_name", "rare_phon", "rare_addr", "ngram"]
n_q = C.set_query_records(con, "SELECT id FROM feat_s1 WHERE hash(id) % 220 = 0")
n_gt = C.load_ground_truth_pairs(con, GT_TSV, s1_filter_sql="SELECT id FROM q")
print(f"pilot: {n_q:,} S1 records, {n_gt:,} true pairs")

def strategy_table(con, strategies, info, loo=False):
    rows = []
    allmask = sum(B[s] for s in strategies)
    union = evaluate_candidate_recall(con)
    for s in strategies:
        e = evaluate_candidate_recall(con, mask=B[s])
        r = {"strategy": s, "recall_%": round(100 * e["recall"], 2), "candidates": e["candidates"], "avg/S1": e["avg_per_s1"],
             "p95/S1": e["p95_per_s1"], "max/S1": e["max_per_s1"], "precision_%": round(100 * e["pair_precision"], 2),
             "seconds": info[s].get("seconds")}
        if loo:
            r["recall_lost_if_removed_pts"] = round(100 * (union["recall"] - evaluate_candidate_recall(con, mask=allmask & ~B[s])["recall"]), 2)
        rows.append(r)
    rows.append({"strategy": "UNION", "recall_%": round(100 * union["recall"], 2), "candidates": union["candidates"],
                 "avg/S1": union["avg_per_s1"], "p95/S1": union["p95_per_s1"], "max/S1": union["max_per_s1"],
                 "precision_%": round(100 * union["pair_precision"], 2),
                 "seconds": round(sum(info[s].get("seconds", 0) for s in strategies), 1)})
    return pd.DataFrame(rows), union

with PeakRSS() as pk:
    info0 = generate_candidates(con, INITIAL_STRATEGIES, INITIAL, max_raw_pairs=50_000_000, log=lambda *a: None)
tab0, union0 = strategy_table(con, INITIAL_STRATEGIES, info0)
display(tab0)
print(f"peak RSS {pk.gb} GB | recall by source: S2 {union0['recall_s2']:.4f}, S3 {union0['recall_s3']:.4f} | by country: {union0['recall_by_country']}")
RESULTS["pilot_initial"] = {"config": asdict(INITIAL), "n_s1": n_q, "true_pairs": n_gt, "table": tab0.to_dict(orient="records"),
                            "union": union0, "peak_rss_gb": pk.gb}

pilot: 9,929 S1 records, 34,587 true pairs


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,strategy,recall_%,candidates,avg/S1,p95/S1,max/S1,precision_%,seconds
0,name,57.41,151719,15.28,77.0,100,13.09,0.84
1,phonetic,57.31,89162,8.98,33.0,50,22.23,0.53
2,structured,66.40,122993,12.39,42.0,92,18.67,0.96
3,rare_name,41.91,94198,9.49,43.0,95,15.39,0.47
4,rare_phon,19.58,59124,5.95,40.0,93,11.45,0.36
5,rare_addr,41.17,92293,9.30,42.0,92,15.43,0.53
6,ngram,24.61,25042,2.52,20.0,20,33.99,9.37
7,UNION,93.81,480502,48.39,111.0,250,6.75,13.10


peak RSS 1.25 GB | recall by source: S2 0.9401, S3 0.9363 | by country: {'india': {'true_pairs': 13979, 'recall': 0.9026}, 'us': {'true_pairs': 20608, 'recall': 0.9622}}


## 5. Failure analysis of the pilot
What do the true pairs missed by the initial union have in common? This decides which strategies to add, rather than raising caps blindly.

In [6]:
con.execute("""CREATE OR REPLACE TEMP TABLE miss AS SELECT g.s1, g.t, g.src FROM gt_pairs g JOIN q ON q.id = g.s1
               WHERE NOT EXISTS (SELECT 1 FROM cand c WHERE c.s1 = g.s1 AND c.t = g.t)""")
con.execute("""CREATE OR REPLACE TEMP TABLE md AS SELECT m.src, a.country, a.name_core a_name, b.name_core b_name,
       a.name_key a_key, b.name_key b_key, a.name_phon a_ph, b.name_phon b_ph, a.name_tokens a_tok, b.name_tokens b_tok,
       a.state a_st, b.state b_st, a.hn a_hn, b.hn b_hn, a.places a_pl, b.places b_pl, a.addr_tokens a_at, b.addr_tokens b_at
     FROM miss m JOIN q a ON a.id = m.s1 JOIN (SELECT * FROM feat_t WHERE id IN (SELECT t FROM miss)) b ON b.id = m.t""")
fa = con.execute("""SELECT COUNT(*) missed_pairs,
   ROUND(100*AVG((country='india')::INT),1) pct_india,
   ROUND(100*AVG((b_st='')::INT),1) pct_target_without_state,
   ROUND(100*AVG((a_st<>b_st AND b_st<>'')::INT),1) pct_state_differs,
   ROUND(100*AVG((a_hn=b_hn AND a_st=b_st AND a_hn<>'')::INT),1) pct_same_state_hn,
   ROUND(100*AVG((len(list_intersect(string_split(a_ph,' '),string_split(b_ph,' ')))>0)::INT),1) pct_share_phonetic_token,
   ROUND(100*AVG((len(list_intersect(string_split(a_tok,' '),string_split(b_tok,' ')))>0)::INT),1) pct_share_name_token,
   ROUND(100*AVG((len(list_intersect(string_split(a_at,' '),string_split(b_at,' ')))>0)::INT),1) pct_share_addr_token
   FROM md""").df()
display(fa.T)
hn_blocks = con.execute("""SELECT quantile_cont(s.n, [0.1, 0.5, 0.9]) FROM md JOIN st_hn s ON s.country = md.country AND s.state = md.a_st
                           AND s.hn = md.a_hn WHERE a_hn = b_hn AND a_st = b_st AND a_hn <> ''""").fetchone()[0]
print("block size (targets) of missed pairs that share state + house number, p10/p50/p90:", hn_blocks)
display(con.execute("SELECT src, country, a_name, b_name, a_st, b_st, a_hn, b_hn, a_pl, b_pl FROM md USING SAMPLE 15 ROWS (reservoir, 7)").df())
RESULTS["pilot_initial_failure_analysis"] = {**fa.iloc[0].to_dict(), "same_state_hn_block_p10_p50_p90": hn_blocks}

,0
missed_pairs,2140.0
pct_india,63.6
pct_target_without_state,28.7
pct_state_differs,5.1
pct_same_state_hn,15.4
pct_share_phonetic_token,85.7
pct_share_name_token,74.1
pct_share_addr_token,69.7


block size (targets) of missed pairs that share state + house number, p10/p50/p90: [144.0, 2801.0, 16497.0]


,src,country,a_name,b_name,a_st,b_st,a_hn,b_hn,a_pl,b_pl
0,2,india,fortune producer,phorchyun prodyusar,mh,mh,,,thane,thane
1,2,us,cardiology care,ccare,oh,oh,1216,,warren,warren
2,2,india,modern investment,modarn investment,tg,ap,52,52,madhura nagar|ameerpet|sanjeev reddy nagar|ameerpet|hyde...,ameerpet|hyderabad
3,2,us,elite invest,elite invest 85209,nc,nc,8906,,concord,concord
4,3,india,star sree consultants,star,mh,mh,,,hutatma chowk|mumbai,hutatma chowk|mumbai
5,3,india,solapur realities,solapur rcalitise,mh,mh,102,,bhawani peth nr ippkayal school|solapur,solapur
6,2,india,universal infrastructure,yunivarsal inphrastrakchar,tg,ap,3,3,vamsirams jyothi granules|serilingampally|kv rangareddy,vamsirams jyothi granules|kv rangareddy|serilingampally
7,2,india,ts packaging,sri tspackaging com,hr,hr,203,203,gurgaon,gurgaon
8,3,us,ng prairie aluminum,ng ng prairie,tx,,6722,,katy,
9,3,india,vision estate,vijan estet,mh,mh,209,209,pune|pune|sambhaji nagar,pune|pune


**What this showed, and what was changed**
* **Targets without a state.** About 29% of the misses are targets whose address is missing or unparsed, so `state = ''`. Every state-partitioned key missed them. The fix is `stateless_fallback`: each state-partitioned key is also looked up in the country's *no-state* bucket, which has its own frequency cap.
* **Common tokens in huge blocks.** Many misses share state + house number, but those blocks hold thousands of targets (median about 2,800 for India). Their names share a phonetic token that is common on its own. The fix is the new **compound** strategy: `(state, house number | place, phonetic token)`, which is specific even when every part is common.
* **Indian transliteration and typo variants.** 86% of the misses share a phonetic token. They're caught by compound keys, by raising the phonetic cap, and by wider n-gram postings (tested in §6).

## 6. Parameter sweep: marginal value of each change
Each variant is scored by what it **adds** to the current union: recall points gained per extra candidate per S1. The base is the initial configuration plus the no-state fallback.

In [7]:
BASE = replace(INITIAL, stateless_fallback=True)
with PeakRSS() as pk:
    info_b = generate_candidates(con, INITIAL_STRATEGIES, BASE, max_raw_pairs=50_000_000, log=lambda *a: None)
con.execute("CREATE OR REPLACE TEMP TABLE cand_base AS SELECT s1, t FROM cand")
b_rec, b_n = con.execute("SELECT (SELECT COUNT(*) FROM gt_pairs g JOIN cand_base USING (s1, t)), (SELECT COUNT(*) FROM cand_base)").fetchone()
print(f"initial + no-state fallback: recall {100*b_rec/n_gt:.2f}% | {b_n/n_q:.1f} candidates per S1 (initial: {100*union0['recall']:.2f}%, {union0['avg_per_s1']})")
VARIANTS = [("rare_name", dict(token_k=3)), ("rare_name", dict(token_df_cap=100)), ("rare_phon", dict(token_k=3)),
            ("rare_addr", dict(token_k=3)), ("rare_addr", dict(addr_token_df_cap=100)),
            ("structured", dict(hn_cap=100, hn_refined_cap=100)), ("structured", dict(hn_cap=200, hn_refined_cap=200)),
            ("name", dict(name_cap=300, name_state_cap=300)), ("phonetic", dict(phon_cap=150, phon_state_cap=150)),
            ("ngram", dict(ngram_min_shared=2)), ("ngram", dict(ngram_top_k=50)), ("ngram", dict(ngram_m=8, ngram_df_cap=1000)),
            ("compound", dict(compound_cap=30)), ("compound", dict(compound_cap=60))]
rows = []
for kind, kw in VARIANTS:
    cfg = replace(BASE, **kw); t = time.time()
    try:
        C._GENERATORS[kind](con, cfg, 50_000_000)
    except C.CandidateExplosion as e:          # the guard refuses to run it: record, do not fail
        rows.append({"strategy": kind, "change": kw, "status": f"stopped: {e}"}); continue
    rec, n = con.execute(f"""WITH u AS (SELECT s1, t FROM cand_base UNION SELECT s1, t FROM cand_{kind})
        SELECT (SELECT COUNT(*) FROM gt_pairs g JOIN u USING (s1, t)), (SELECT COUNT(*) FROM u)""").fetchone()
    rows.append({"strategy": kind, "change": kw, "status": "ok", "union_recall_%": round(100*rec/n_gt, 2), "gain_pts": round(100*(rec-b_rec)/n_gt, 2),
                 "extra_cands_per_s1": round((n-b_n)/n_q, 1), "seconds": round(time.time()-t, 1)})
    con.execute(f"DROP TABLE IF EXISTS cand_{kind}")
sweep = pd.DataFrame(rows); sweep["gain_per_10_cands"] = (10 * sweep.gain_pts / sweep.extra_cands_per_s1.clip(lower=0.1)).round(3)
display(sweep)
print("peak RSS", pk.gb, "GB")
RESULTS["sweep"] = {"base": {"recall": round(b_rec/n_gt, 4), "avg_per_s1": round(b_n/n_q, 1)}, "variants": sweep.to_dict(orient="records")}

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

initial + no-state fallback: recall 94.84% | 60.1 candidates per S1 (initial: 93.81%, 48.39)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,strategy,change,status,union_recall_%,gain_pts,extra_cands_per_s1,seconds,gain_per_10_cands
0,rare_name,{'token_k': 3},ok,94.84,0.00,0.4,0.5,0.000
1,rare_name,{'token_df_cap': 100},ok,95.22,0.38,29.0,0.5,0.131
2,rare_phon,{'token_k': 3},ok,94.84,0.00,0.1,0.5,0.000
3,rare_addr,{'token_k': 3},ok,94.85,0.01,0.6,0.8,0.167
4,rare_addr,{'addr_token_df_cap': 100},ok,95.36,0.52,11.4,0.6,0.456
5,structured,"{'hn_cap': 100, 'hn_refined_cap': 100}",ok,95.04,0.20,10.2,1.0,0.196
6,structured,"{'hn_cap': 200, 'hn_refined_cap': 200}",ok,95.22,0.38,29.5,1.0,0.129
7,name,"{'name_cap': 300, 'name_state_cap': 300}",ok,94.89,0.04,9.3,0.6,0.043
8,phonetic,"{'phon_cap': 150, 'phon_state_cap': 150}",ok,95.06,0.22,7.2,0.6,0.306
9,ngram,{'ngram_min_shared': 2},ok,95.06,0.22,5.5,10.1,0.400


peak RSS 0.51 GB


**Decisions from the sweep**
* **Kept:** the compound strategy (by far the best recall per candidate), the phonetic cap of 150, an address-token cap of 100, state + house-number caps of 100, and n-gram postings with `m = 8` and a cap of 1000.
* **Not kept:** more rare tokens per record (`k = 3`, no gain); any variant the explosion guard refused to run (shown as `stopped` above); a name cap of 300 (+0.04 points for about 9 candidates per S1); n-gram `top_k = 50` (no gain); n-gram `min_shared = 2`, which adds about 5 candidates per S1 for 0.2 points and is dominated by the wider postings.

These values are now the defaults of `BlockingConfig`.

## 7. Final configuration on the 10k pilot

In [8]:
FINAL = replace(BlockingConfig(), per_s1_cap=None)
ALL_EVALUATED = list(DEFAULT_STRATEGIES) + ["rare_phon"]
with PeakRSS() as pk:
    info_f = generate_candidates(con, ALL_EVALUATED, FINAL, max_raw_pairs=50_000_000, log=lambda *a: None)
tab_f, union_f = strategy_table(con, ALL_EVALUATED, info_f, loo=True)
display(tab_f)
print(f"peak RSS {pk.gb} GB")

# greedy forward selection by recall gained per added candidate (from the strategy bitmask)
con.execute("CREATE OR REPLACE TEMP TABLE cg AS SELECT c.mask, (g.s1 IS NOT NULL) hit FROM cand c LEFT JOIN gt_pairs g USING (s1, t)")
ev = lambda m: con.execute(f"SELECT COUNT(*) FILTER (WHERE hit), COUNT(*) FROM cg WHERE mask & {m} > 0").fetchone()
m, left, steps = 0, list(ALL_EVALUATED), []
while left:
    r0, n0 = ev(m) if m else (0, 0)
    gain = {s: ev(m | B[s]) for s in left}
    best = max(left, key=lambda s: (gain[s][0] - r0) / max(gain[s][1] - n0, 1))
    m |= B[best]; left.remove(best); r, n = gain[best]
    steps.append({"add": best, "cumulative_recall_%": round(100*r/n_gt, 2), "avg_cands_per_s1": round(n/n_q, 1)})
greedy = pd.DataFrame(steps); display(greedy)

# per-S1 cap
caps = []
con.execute("CREATE OR REPLACE TEMP TABLE cand_full AS SELECT * FROM cand WHERE mask & " + str(sum(B[s] for s in DEFAULT_STRATEGIES)) + " > 0")
for cap in (None, 300, 200, 150, 100):
    con.execute("CREATE OR REPLACE TABLE cand AS SELECT * FROM cand_full")
    if cap: C.apply_per_s1_cap(con, cap, CAP_PRIORITY)
    e = evaluate_candidate_recall(con)
    caps.append({"per_s1_cap": cap or "none", "recall_%": round(100*e["recall"], 2), "avg/S1": e["avg_per_s1"], "p95/S1": e["p95_per_s1"],
                 "max/S1": e["max_per_s1"], "recall_india_%": round(100*e["recall_by_country"]["india"]["recall"], 2),
                 "recall_us_%": round(100*e["recall_by_country"]["us"]["recall"], 2)})
caps = pd.DataFrame(caps); display(caps)
RESULTS["pilot_final"] = {"config": asdict(BlockingConfig()), "table": tab_f.to_dict(orient="records"), "union_all_evaluated": union_f,
                          "greedy": steps, "per_s1_cap": caps.to_dict(orient="records"), "peak_rss_gb": pk.gb}

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,strategy,recall_%,candidates,avg/S1,p95/S1,max/S1,precision_%,seconds,recall_lost_if_removed_pts
0,compound,79.42,165007,16.62,44.0,112,16.65,4.93,1.18
1,ngram,50.08,138617,13.96,20.0,20,12.49,11.70,0.37
2,name,57.61,159882,16.10,78.0,119,12.46,0.68,0.31
3,rare_addr,51.46,211441,21.30,88.0,189,8.42,0.68,1.16
4,structured,67.16,224437,22.60,83.0,163,10.35,1.20,1.42
5,phonetic,60.01,223758,22.54,104.0,269,9.28,0.63,0.59
6,rare_name,44.43,159656,16.08,61.0,148,9.63,0.54,0.26
7,rare_phon,20.83,108051,10.88,56.0,146,6.67,0.46,0.00
8,UNION,97.33,1075700,108.34,216.0,425,3.13,20.80,NaN


peak RSS 1.32 GB


,add,cumulative_recall_%,avg_cands_per_s1
0,compound,79.42,16.6
1,ngram,87.58,28.6
2,name,91.49,41.9
3,rare_addr,94.82,60.9
4,structured,96.34,80.3
5,phonetic,96.99,92.2
6,rare_name,97.33,103.8
7,rare_phon,97.33,108.3


,per_s1_cap,recall_%,avg/S1,p95/S1,max/S1,recall_india_%,recall_us_%
0,none,97.33,103.80,213.0,425,95.39,98.65
1,300,97.33,103.68,213.0,300,95.38,98.65
2,200,97.23,101.31,200.0,200,95.17,98.62
3,150,96.98,95.19,150.0,150,94.68,98.54
4,100,96.07,79.26,100.0,100,92.82,98.27


**Chosen combination:** `compound, ngram, name, rare_addr, structured, phonetic, rare_name`, with a per-S1 cap of 200.

**Rejected:** `rare_phon`. Its leave-one-out loss is 0.00 points and it costs about 11 candidates per S1, because its matches are already covered by compound and phonetic keys.

In [9]:
print("Remaining misses of the final configuration — examples")
con.execute("CREATE OR REPLACE TABLE cand AS SELECT * FROM cand_full")
con.execute("""CREATE OR REPLACE TEMP TABLE miss2 AS SELECT g.s1, g.t, g.src FROM gt_pairs g JOIN q ON q.id = g.s1
               WHERE NOT EXISTS (SELECT 1 FROM cand c WHERE c.s1 = g.s1 AND c.t = g.t)""")
fails = con.execute("""SELECT m.src, a.country, a.name_core s1_name, b.name_core target_name, a.state s1_st, b.state t_st,
        a.hn s1_hn, b.hn t_hn, a.places s1_places, b.places t_places
        FROM miss2 m JOIN q a ON a.id = m.s1 JOIN (SELECT * FROM feat_t WHERE id IN (SELECT t FROM miss2)) b ON b.id = m.t
        USING SAMPLE 20 ROWS (reservoir, 3)""").df()
display(fails)
fa2 = con.execute("""SELECT COUNT(*) n, ROUND(100*AVG((a.country='india')::INT),1) pct_india,
        ROUND(100*AVG((b.state='')::INT),1) pct_target_without_state,
        ROUND(100*AVG((a.state<>b.state AND b.state<>'')::INT),1) pct_state_differs,
        ROUND(100*AVG((len(list_intersect(string_split(a.name_tokens,' '), string_split(b.name_tokens,' '))) = 0
                       AND len(list_intersect(string_split(a.name_phon,' '), string_split(b.name_phon,' '))) = 0)::INT),1) pct_no_shared_name_or_phon_token
        FROM miss2 m JOIN q a ON a.id = m.s1 JOIN (SELECT * FROM feat_t WHERE id IN (SELECT t FROM miss2)) b ON b.id = m.t""").df()
display(fa2.T)
RESULTS["pilot_final_failure_analysis"] = fa2.iloc[0].to_dict()

Remaining misses of the final configuration — examples


,src,country,s1_name,target_name,s1_st,t_st,s1_hn,t_hn,s1_places,t_places
0,3,india,great ventures,gret bhenycharas,wb,wb,1,1,kolkata|kolkata|howrah,howrah
1,3,india,systems nirala foods of delhi,iridova,dl,dl,100,116,ganesh nagar ii|delhi|e delhi,e delhi|delhi
2,3,india,seven services,sevan sarvisis,gj,gj,,,chandkheda|ahmedabad,ahmedabad
3,2,india,guru software,guru saphatoyyar,wb,wb,101,48,shanti bhawan|kolkata|howrah|kolkata,howrah|kolkata
4,2,india,sunrise maa logistics,sanrais ma lajistiks,tn,tn,1,901,ambattur|chennai,chennai
5,2,india,southern business,sadarn bijnes,up,up,5,4,ghaziabad,ghaziabad
6,2,india,vision arihant producer,vishan arihant purotyusar,tn,tn,1,1,valluvar salai|saidapet|kanchipuram,chennai|kanchipuram
7,3,india,shyam solutions,shyam limited private solutions,mh,mh,,15,mumbai|goandevi dongr andheri|mumbai,mumbai
8,3,india,shastra seva samiti,shastraseva,kl,kl,2,2,edakkandi|vallikkunnu po|malappuram|malappuram|malapuram,malappuram|malapuram
9,3,india,vision investment,viprivate,ka,ka,216,216,ramanashree arcade|bangalore,eglr|bangalore


,0
n,923.0
pct_india,69.9
pct_target_without_state,21.3
pct_state_differs,11.7
pct_no_shared_name_or_phon_token,21.8


## 8. 100k validation (chunked)
The sample is `hash(id) % 22 = 0`, about 100k S1 records. Generation runs in 10 hash chunks of about 10k, so each chunk has the same bounded memory and raw-pair profile as the pilot. Disk is checked before every chunk.

This runs in a separate working database, which is deleted afterwards.

In [10]:
# free the pilot's scratch tables and release the statistics DB (DuckDB allows one handle per file)
N_S1 = con.execute("SELECT COUNT(*) FROM feat_s1").fetchone()[0]
for t in [r[0] for r in con.execute("SELECT table_name FROM duckdb_tables() WHERE database_name = current_database() AND table_name NOT LIKE 'st_%'").fetchall()]:
    con.execute(f"DROP TABLE IF EXISTS {t}")
con.execute("CHECKPOINT"); con.close()
work = P / "work_100k.duckdb"
if work.exists(): work.unlink()
cw = C.connect(work, memory_limit="1500MB", threads=4, temp_dir=P / "duckdb_tmp", max_temp="2GB")
cw.execute(f"ATTACH '{P / 'blocking_stats.duckdb'}' AS st (READ_ONLY)")
for t in [r[0] for r in cw.execute("SELECT table_name FROM duckdb_tables() WHERE database_name = 'st' AND table_name LIKE 'st_%'").fetchall()]:
    cw.execute(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM st.{t}")
C.register_features(cw, P)
cw.execute("CREATE OR REPLACE TABLE q_all AS SELECT * FROM feat_s1 WHERE hash(id) % 22 = 0")
n100 = cw.execute("SELECT COUNT(*) FROM q_all").fetchone()[0]
g100 = C.load_ground_truth_pairs(cw, GT_TSV, s1_filter_sql="SELECT id FROM q_all")
print(f"100k validation: {n100:,} S1 records, {g100:,} true pairs")
free_before = C.check_disk(P, MIN_FREE_GB)
with PeakRSS() as pk:
    run100 = generate_candidates_chunked(cw, DEFAULT_STRATEGIES, replace(BlockingConfig(), per_s1_cap=None), n_chunks=10,
                                         min_free_gb=MIN_FREE_GB, log=print)
tab100, u100 = strategy_table(cw, list(DEFAULT_STRATEGIES), {s: {"seconds": v["seconds"]} for s, v in run100["per_strategy"].items()}, loo=True)
display(tab100)
C.apply_per_s1_cap(cw, 200, CAP_PRIORITY)
u100c = evaluate_candidate_recall(cw)
summary100 = pd.DataFrame([{"setting": "uncapped", **{k: u100[k] for k in ("recall", "recall_s2", "recall_s3", "candidates", "avg_per_s1", "p95_per_s1", "p99_per_s1", "max_per_s1", "pct_s1_without_candidates")}},
                           {"setting": "per-S1 cap 200 (final)", **{k: u100c[k] for k in ("recall", "recall_s2", "recall_s3", "candidates", "avg_per_s1", "p95_per_s1", "p99_per_s1", "max_per_s1", "pct_s1_without_candidates")}}])
display(summary100)
print("recall by country (capped):", u100c["recall_by_country"])
work_gb = round(work.stat().st_size / 1e9, 2)
print(f"total {run100['seconds']}s | peak RSS {pk.gb} GB | work DB {work_gb} GB | free disk before {free_before:.1f} GB")
RESULTS["validation_100k"] = {"n_s1": n100, "true_pairs": g100, "run": run100, "table": tab100.to_dict(orient="records"),
                              "union_uncapped": u100, "union_cap200": u100c, "peak_rss_gb": pk.gb, "work_db_gb": work_gb}

100k validation: 99,514 S1 records, 344,494 true pairs


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

chunk 1/10: 1,015,515 pairs in 22s


...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

chunk 10/10: 1,032,293 pairs in 29s


,strategy,recall_%,candidates,avg/S1,p95/S1,max/S1,precision_%,seconds,recall_lost_if_removed_pts
0,compound,79.56,1653446,16.62,44.0,135,16.58,59.4,1.20
1,ngram,49.94,1383038,13.90,20.0,20,12.44,153.9,0.40
2,name,57.86,1615635,16.24,78.0,119,12.34,8.5,0.31
3,rare_addr,51.26,2116453,21.27,88.0,194,8.34,8.5,1.12
4,structured,67.68,2228998,22.40,82.0,252,10.46,13.1,1.46
5,phonetic,60.00,2273096,22.84,105.0,277,9.09,6.6,0.60
6,rare_name,44.57,1586154,15.94,61.0,184,9.68,6.7,0.36
7,UNION,97.58,10311647,103.62,212.0,635,3.26,256.7,NaN


,setting,recall,recall_s2,recall_s3,candidates,avg_per_s1,p95_per_s1,p99_per_s1,max_per_s1,pct_s1_without_candidates
0,uncapped,0.9758,0.9775,0.9742,10311647,103.62,212.0,270.0,635,0.0
1,per-S1 cap 200 (final),0.9746,0.9766,0.9728,10063806,101.13,200.0,200.0,200,0.0


recall by country (capped): {'india': {'true_pairs': 137528, 'recall': 0.9552}, 'us': {'true_pairs': 206966, 'recall': 0.9875}}
total 259.4s | peak RSS 1.6 GB | work DB 0.32 GB | free disk before 6.8 GB


## 9. Full-scale execution path (designed and estimated; **not executed** here)
Every S1 record goes through the same chunked function, with `out_parquet_dir` set. Each chunk writes one Parquet part of `(s1 BIGINT, t BIGINT, mask USMALLINT)`, and the per-S1 cap is applied inside the chunk. There is never one giant in-memory table.

The estimate below extrapolates **linearly** from the 100k run: time per S1 and bytes per candidate pair.

In [11]:
pairs_per_s1 = u100c["avg_per_s1"]
cw.execute(f"COPY (SELECT * FROM cand) TO '{P / 'cand_100k_sample.parquet'}' (FORMAT parquet, COMPRESSION zstd)")
bytes_per_pair = (P / "cand_100k_sample.parquet").stat().st_size / u100c["candidates"]
(P / "cand_100k_sample.parquet").unlink()
sec_per_s1 = run100["seconds"] / n100
est = {"s1_records": N_S1, "est_pairs": int(N_S1 * pairs_per_s1), "est_parquet_gb": round(N_S1 * pairs_per_s1 * bytes_per_pair / 1e9, 2),
       "bytes_per_pair_parquet": round(bytes_per_pair, 2), "est_hours_4_threads": round(N_S1 * sec_per_s1 / 3600, 2),
       "suggested_chunks": int(N_S1 / 10_000)}
display(pd.Series(est).to_frame("full-scale estimate").T)
RESULTS["full_scale_estimate"] = est

,s1_records,est_pairs,est_parquet_gb,bytes_per_pair_parquet,est_hours_4_threads,suggested_chunks
full-scale estimate,2206821.0,223175807.0,1.04,4.65,1.6,220.0


**Full-scale command** (not run in this notebook):
```python
con.execute("CREATE TABLE q_all AS SELECT * FROM feat_s1")
generate_candidates_chunked(con, DEFAULT_STRATEGIES, BlockingConfig(), n_chunks=220,
                            out_parquet_dir="data/processed/phase3/candidates", min_free_gb=4)
```

## 10. Save results and clean up

In [12]:
cw.close(); (P / "work_100k.duckdb").unlink(missing_ok=True); Path(str(P / "work_100k.duckdb") + ".wal").unlink(missing_ok=True)
shutil.rmtree(P / "duckdb_tmp", ignore_errors=True)
RESULTS["disk_after_gb_free"] = round(C.check_disk(P, 0), 1)
RESULTS["phase3_artifacts_mb"] = {p.name: round(sum(f.stat().st_size for f in (p.rglob("*") if p.is_dir() else [p])) / 1e6, 1)
                                  for p in P.iterdir()}
RESULTS["notebook_finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
def _clean(o):
    import math, numpy as np
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if math.isnan(o) else round(float(o), 4)
    if isinstance(o, np.bool_): return bool(o)
    return o
(EXP / "phase3_candidate_generation_results.json").write_text(json.dumps(_clean(RESULTS), indent=2))
print("wrote experiments/phase3_candidate_generation_results.json")
display(pd.Series(RESULTS["phase3_artifacts_mb"], name="MB").to_frame().T)

wrote experiments/phase3_candidate_generation_results.json


,features_s1,blocking_stats.duckdb,features_s2,features_s3,materialize_stats.json
MB,163.2,349.7,384.4,401.6,0.0
